In [1]:
#!/usr/bin/env python3
"""
Deep Past Challenge — Akkadian-to-English Translation  v20
===========================================================
TWO-MODEL ENSEMBLE  |  Cross-model-aware agreement MBR
===========================================================

SCORE HISTORY:
  v13  34.3  single-model, composite MBR
  v16  34.8  two-model, pure chrF++ MBR
  v17  35.4  two-model, geo-mean(BLEU×chrF++) MBR       ← was called v18
  35.5 comp  two-model, geo-mean + uniform agreement bonus
  v19  ???   our pipeline + competitor's agreement bonus  ← parallel run
  v20  ???   cross-model-aware agreement bonus            ← THIS

═══════════════════════════════════════════════════════════════
WHAT v20 ADDS: SOURCE-AWARE AGREEMENT SCORING
═══════════════════════════════════════════════════════════════

The 35.5 competitor applies the same agreement bonus regardless
of whether candidates agree within one model or across models.
But these two types of agreement carry fundamentally different
amounts of information:

  WITHIN-MODEL agreement (beam-rank-1 = beam-rank-2, same model):
    Adjacent beam paths differ by tiny token-level perturbations.
    High correlation. Weak evidence of ground-truth correctness.

  CROSS-MODEL agreement (model A output = model B output):
    Two independently fine-tuned checkpoints trained on different
    random seeds converge on the exact same English string via
    completely different generation paths. This is strong evidence
    that the string is high-probability under both models'
    independently learned distributions.
    
Literature basis:
  Sennrich et al. (2016) "Edinburgh NMT": cross-model agreement
  as ensemble confidence signal. Wang et al. (NeurIPS 2023)
  "Self-consistency improves CoT reasoning": the most-agreed-upon
  answer across independent sampling chains is more likely correct.
  The key word is INDEPENDENT — cross-model chains are maximally
  independent. Eikema & Aziz (EMNLP 2022): frequency under the
  model approximates probability; cross-model frequency approximates
  probability under the product of distributions (product-of-experts),
  which is sharper and more accurate.

SCORING FORMULA:
  score(h_i) = count_weighted_geomean_utility(h_i)
             + within_model_bonus × max(0, within_count(h_i) - 1)
             + cross_model_bonus  × cross_model_agreed(h_i)

  where:
    within_count(h_i)     = times h_i appears in pool of SAME model
    cross_model_agreed(h) = 1 if both model A AND B produced h_i, else 0
    within_model_bonus    = 0.015  (small — weak evidence)
    cross_model_bonus     = 0.12   (strong — independent confirmation)

  A candidate that both models produce gets +0.12, which is 3.4×
  the competitor's flat 0.035. A candidate seen twice in one model
  but not the other gets only +0.015 (less than half of competitor).

This is the theoretically correct weighting: it calibrates the
bonus to the actual information content of the agreement type.

═══════════════════════════════════════════════════════════════
GENERATION CHANGES OVER v19
═══════════════════════════════════════════════════════════════
  epsilon samples: 4 → 6 (more cross-model collision opportunities)
  pool_cap:       36 → 40
  Everything else identical to v19.

  Pool: 4 beam + 6 nucleus + 6 ε = 16/model × 2 = ~32 total
"""

from __future__ import annotations

import gc, json, logging, math, os, random, re, subprocess, importlib, sys
from contextlib import nullcontext
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR   = Path("/kaggle/working")

# ── Path discovery ──────────────────────────────────────────────────────────
print("=" * 65)
for root, dirs, files in os.walk(INPUT_ROOT):
    depth = root.replace(str(INPUT_ROOT), "").count(os.sep)
    print(f"{'  '*depth}{Path(root).name}/")
    for fn in sorted(files):
        sz = os.path.getsize(os.path.join(root, fn))
        print(f"{'  '*(depth+1)}{fn}  "
              f"({'%.1f'%(sz/1e9)+' GB' if sz>1e8 else str(sz//1024)+' KB'})")
print("=" * 65)

def _find_file(root, *names):
    for dp, _, fns in os.walk(root):
        for n in names:
            if n in fns: return Path(dp) / n
    return None

def _find_whl_dirs(root):
    s = set()
    for dp, _, fns in os.walk(root):
        if any(f.endswith(".whl") for f in fns): s.add(Path(dp))
    return list(s)

def _find_model_dirs(root) -> List[Path]:
    """Find ALL HuggingFace checkpoints; sort by safetensors size; guard identity."""
    MARKERS = {"config.json", "pytorch_model.bin", "model.safetensors"}
    found: List[Path] = []
    for dp, _, files in os.walk(root):
        if MARKERS & set(files):
            p = Path(dp)
            if p not in found: found.append(p)
    def _sf_size(p: Path) -> int:
        sf = p / "model.safetensors"
        return sf.stat().st_size if sf.exists() else 0
    found.sort(key=_sf_size, reverse=True)
    if (len(found) >= 2
            and "mbr" in str(found[0]).lower()
            and "optimized" not in str(found[0]).lower()):
        found[0], found[1] = found[1], found[0]
    return found

TEST_FILE  = _find_file(INPUT_ROOT, "test.csv", "test_sentences.csv")
WHL_DIRS   = _find_whl_dirs(INPUT_ROOT)
ALL_MODELS = _find_model_dirs(INPUT_ROOT)

if not TEST_FILE:  raise FileNotFoundError("test.csv not found.")
if not ALL_MODELS: raise FileNotFoundError("No model checkpoint found.")

MODEL_A        = str(ALL_MODELS[0])
MODEL_B        = str(ALL_MODELS[1]) if len(ALL_MODELS) > 1 else None
TWO_MODEL_MODE = MODEL_B is not None

print("\n" + "═"*65)
print("  MODEL DISCOVERY")
print("═"*65)
for i, mp in enumerate(ALL_MODELS):
    sf = mp / "model.safetensors"
    gb = sf.stat().st_size / 1e9 if sf.exists() else 0
    print(f"  {'MODEL_A (base)' if i==0 else 'MODEL_B (ensemble)'}: "
          f"{mp.relative_to(INPUT_ROOT)}  ({gb:.2f} GB)")
if not TWO_MODEL_MODE:
    print("  ⚠️  WARNING: Single-model — attach mattiaangeli/byt5-akkadian-mbr")
    print("  NOTE: Cross-model agreement bonus requires two models.")
    print("        Score will fall back to within-model-only agreement.")
print("═"*65)

# ── Offline packages ─────────────────────────────────────────────────────────
def _pkg_ok(n):
    try: importlib.import_module(n); return True
    except ImportError: return False

def _try_install(pkg):
    for d in WHL_DIRS:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                            "--no-index", f"--find-links={d}", pkg],
                           capture_output=True, text=True)
        if r.returncode == 0: importlib.invalidate_caches(); return True
    return False

for pkg in ("portalocker", "sacrebleu"):
    if not _pkg_ok(pkg):
        ok = _try_install(pkg)
        print(f"  {'✅' if ok else '⚠️'} {pkg} {'installed' if ok else 'fallback'}")

try:
    import sacrebleu as _sacrebleu; SACREBLEU_OK = True; print("✅ sacrebleu ready.")
except ImportError:
    SACREBLEU_OK = False; print("⚠️ sacrebleu fallback.")

# ── Pure-Python fallback metrics ─────────────────────────────────────────────
def _ngrams_c(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def _py_chrf(a, b, char_order=6, word_order=2):
    if not a or not b: return 0.0
    ps, rs = [], []
    for cn in range(1, char_order+1):
        hg = Counter(a[i:i+cn] for i in range(len(a)-cn+1))
        rg = Counter(b[i:i+cn] for i in range(len(b)-cn+1))
        m = sum(min(c, rg[g]) for g, c in hg.items())
        ps.append(m/max(1, sum(hg.values()))); rs.append(m/max(1, sum(rg.values())))
    for wn in range(1, word_order+1):
        hg = _ngrams_c(a.split(), wn); rg = _ngrams_c(b.split(), wn)
        m = sum(min(c, rg[g]) for g, c in hg.items())
        ps.append(m/max(1, sum(hg.values()))); rs.append(m/max(1, sum(rg.values())))
    ap = sum(ps)/len(ps); ar = sum(rs)/len(rs); d = 4*ap + ar
    return 100.0*5*ap*ar/d if d > 0 else 0.0

def _py_sbleu(hyp, ref):
    if not hyp or not ref: return 0.0
    hw, rw = hyp.split(), ref.split()
    if not hw or not rw: return 0.0
    bp = 1.0 if len(hw) >= len(rw) else math.exp(1 - len(rw)/max(1, len(hw)))
    log_avg = 0.0; max_n = min(4, len(hw), len(rw))
    if max_n == 0: return 0.0
    for n in range(1, max_n+1):
        hg = _ngrams_c(hw, n); rg = _ngrams_c(rw, n)
        m = sum(min(c, rg[g]) for g, c in hg.items())
        num = m+1 if n > 1 else m; den = max(1, len(hw)-n+1) + (1 if n > 1 else 0)
        log_avg += math.log(max(num/den, 1e-10))
    return 100.0 * bp * math.exp(log_avg/max_n)

# ── Device / BF16 ─────────────────────────────────────────────────────────────
def _cuda_bf16():
    if not torch.cuda.is_available(): return False
    try: return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except: return False

def _bf16_ctx(device, enabled):
    if enabled and device.type == "cuda" and _cuda_bf16():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = _cuda_bf16()
print(f"  Device: {DEVICE}"+(f" / {torch.cuda.get_device_name(0)}"
      f" / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
      if torch.cuda.is_available() else ""))

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

# ── Config ────────────────────────────────────────────────────────────────────
@dataclass
class Config:
    model_a_path:   str = MODEL_A
    model_b_path:   str = MODEL_B
    test_data_path: str = str(TEST_FILE)
    output_dir:     str = str(WORK_DIR)

    max_input_length: int = 512
    max_new_tokens:   int = 384
    batch_size:       int = 2
    num_workers:      int = 2
    num_buckets:      int = 6

    # v13-proven generation params — DO NOT CHANGE
    num_beam_cands:     int   = 4
    num_beams:          int   = 8
    length_penalty:     float = 1.3
    repetition_penalty: float = 1.2
    early_stopping:     bool  = True
    # no_repeat_ngram_size ABSENT (Akkadian formulaic repetitions must survive)

    # Multi-temperature nucleus
    use_sampling:        bool        = True
    sample_temperatures: List[float] = field(
        default_factory=lambda: [0.60, 0.80, 0.75])  # competitor's T=0.75 included
    num_sample_per_temp: int   = 2
    top_p:               float = 0.92

    # Epsilon ε=0.02 — increased to 6 for more cross-model collision opportunities
    use_epsilon: bool  = True
    epsilon:     float = 0.02
    num_epsilon: int   = 6   # v19: 4 → v20: 6

    # ── v20: cross-model-aware agreement bonuses ─────────────────────────
    # Cross-model: both models independently produced the same string.
    # This is ~4× more informative than within-model repetition.
    cross_model_bonus: float = 0.12   # strong: two independent distributions agreed
    within_model_bonus: float = 0.015  # weak: same model, correlated beam/sample paths
    # Count-weighted utility (Eikema & Aziz 2022 importance-sampled MBR)
    use_weighted_utility: bool = True
    mbr_pool_cap:         int  = 40   # larger pool for richer collision detection

    use_bf16_amp:           bool = USE_BF16
    use_better_transformer: bool = True
    use_bucket_batching:    bool = True
    use_adaptive_beams:     bool = True
    checkpoint_freq:        int  = 200

    @property
    def num_sample_cands(self): return len(self.sample_temperatures)*self.num_sample_per_temp
    @property
    def num_eps_cands(self): return self.num_epsilon if self.use_epsilon else 0
    @property
    def cands_per_model(self): return self.num_beam_cands+self.num_sample_cands+self.num_eps_cands

    def __post_init__(self):
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        assert self.num_beams >= self.num_beam_cands

# ── Logging ───────────────────────────────────────────────────────────────────
def setup_logging(output_dir, version):
    for h in logging.root.handlers[:]: logging.root.removeHandler(h)
    logging.basicConfig(level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[logging.StreamHandler(),
                  logging.FileHandler(Path(output_dir)/f"inference_{version}.log")])
    return logging.getLogger(f"v{version}")

# ── Preprocessing — CORRECT diacritics (sz→š, not sz→sh) ─────────────────────
_V2=re.compile(r"([aAeEiIuU])(?:2|₂)"); _V3=re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE=str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE=str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})
def _diacritics(s):
    s=s.replace("sz","š").replace("SZ","Š").replace("s,","ṣ").replace("S,","Ṣ")
    s=s.replace("t,","ṭ").replace("T,","Ṭ")
    s=_V2.sub(lambda m:m.group(1).translate(_ACUTE),s)
    return _V3.sub(lambda m:m.group(1).translate(_GRAVE),s)

_FRACS=[(1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),(1/2,"0.5"),
        (2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333")]
_FRAC_TOL=2e-3; _FLOAT_RE=re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
def _canon_dec(x):
    ip=int(math.floor(x+1e-12)); frac=x-ip
    best=min(_FRACS,key=lambda t:abs(frac-t[0]))
    if abs(frac-best[0])<=_FRAC_TOL:
        dec=best[1]; return dec if ip==0 else (f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}")
    return f"{x:.5f}".rstrip("0").rstrip(".")

_WS_RE=re.compile(r"\s+")
_GAP_RE=re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)|\(\s*\d+\s+broken\s+lines?\s*\)",re.I)
def _ngaps(s): return s.str.replace(_GAP_RE,"<gap>",regex=True)
_CT=str.maketrans({"ḫ":"h","Ḫ":"H","ʾ":"","₀":"0","₁":"1","₂":"2","₃":"3",
    "₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9","—":"-","–":"-"})
_U_UP=r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_U_LO=r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DU=re.compile(r"\((["+_U_UP+r"0-9]{1,6})\)"); _DL=re.compile(r"\((["+_U_LO+r"]{1,4})\)")
_PN_RE=re.compile(r"\bPN\b"); _KB_RE=re.compile(r"KÙ\.B\.")
_EF_RE=re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EF_MAP={"0.8333":"⅚","0.6666":"⅔","0.3333":"⅓","0.1666":"⅙","0.625":"⅝","0.75":"¾","0.25":"¼","0.5":"½"}
def _fr(m): return _EF_MAP[m.group(0)]

class OptimizedPreprocessor:
    def preprocess_batch(self,texts):
        s=pd.Series(texts).fillna("").astype(str).apply(_diacritics)
        s=s.str.replace(_DU,r"\1",regex=True).str.replace(_DL,r"{\1}",regex=True)
        s=_ngaps(s).str.translate(_CT).str.replace("ₓ","",regex=False)
        s=s.str.replace(_KB_RE,"KÙ.BABBAR",regex=True).str.replace(_EF_RE,_fr,regex=True)
        s=s.str.replace(_FLOAT_RE,lambda m:_canon_dec(float(m.group(1))),regex=True)
        return s.str.replace(_WS_RE," ",regex=True).str.strip().tolist()

# ── Postprocessing — all 7 fixes, () PRESERVED ───────────────────────────────
_SG_RE=re.compile(r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)(?:\.\s*(?:plur|plural|sing|singular))?\.?\s*[^)]*\)",re.I)
_BG_RE=re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*",re.I)
_UNC_RE=re.compile(r"\(\?\)"); _CDQ=re.compile("[\u201c\u201d]"); _CSQ=re.compile("[\u2018\u2019]")
_MO_RE=re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b",re.I)
_R2I={"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
_RW_RE=re.compile(r"\b(\w+)(?:\s+\1\b)+"); _RP_RE=re.compile(r"([.,])\1+"); _PS_RE=re.compile(r"\s+([.,:])")
_FORB=str.maketrans("","","——<>⌈⌋⌊[]+ʾ;")  # () intentionally excluded — appear in ground truth
_CM_RE=re.compile(r'(?<=\s)-(gold|tax|textiles)\b')
_CR={"gold":"pašallum gold","tax":"šadduātum tax","textiles":"kutānum textiles"}
def _cm(m): return _CR[m.group(1)]
_SK=[(re.compile(r'5\s+11\s*/\s*12\s+shekels?',re.I),'6 shekels less 15 grains'),
     (re.compile(r'5\s*/\s*12\s+shekels?',re.I),'⅓ shekel 15 grains'),
     (re.compile(r'7\s*/\s*12\s+shekels?',re.I),'½ shekel 15 grains'),
     (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?',re.I),'15 grains')]
_SA_RE=re.compile(r'(?<![0-9/])\s+/\s+(?![0-9])\S+')
_ST_RE=re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>'); _ES_RE=re.compile(r'(?<!\w)(?:\.\.+|xx+)(?!\w)')
_MG_RE=re.compile(r'(?:<gap>\s*){2,}'); _HK_TR=str.maketrans({"ḫ":"h","Ḫ":"H"})
def _mon(m): return f"Month {_R2I.get(m.group(1).upper(),m.group(1))}"

class VectorizedPostprocessor:
    def postprocess_batch(self,texts):
        s=pd.Series(texts).fillna("").astype(str)
        s=_ngaps(s).str.replace(_PN_RE,"<gap>",regex=True).str.replace(_CM_RE,_cm,regex=True)
        for pat,repl in _SK: s=s.str.replace(pat,repl,regex=True)
        s=s.str.replace(_EF_RE,_fr,regex=True)
        s=s.str.replace(_FLOAT_RE,lambda m:_canon_dec(float(m.group(1))),regex=True)
        s=s.str.replace(_SG_RE," ",regex=True).str.replace(_BG_RE," ",regex=True)
        s=s.str.replace(_UNC_RE,"",regex=True).str.replace(_ST_RE,"",regex=True)
        s=s.str.replace(_ES_RE,"",regex=True).str.replace(_SA_RE,"",regex=True)
        s=s.str.replace(_CDQ,'"',regex=True).str.replace(_CSQ,"'",regex=True)
        s=s.str.replace(_MO_RE,_mon,regex=True).str.replace(_MG_RE,"<gap>",regex=True)
        s=s.str.replace("<gap>","\x00G\x00",regex=False).str.translate(_FORB)
        s=s.str.replace("\x00G\x00"," <gap> ",regex=False).str.translate(_HK_TR)
        s=s.str.replace(_RW_RE,r"\1",regex=True)
        for n in range(4,1,-1):
            s=s.str.replace(r"\b((?:\w+\s+){"+str(n-1)+r"}\w+)(?:\s+\1\b)+",r"\1",regex=True)
        s=s.str.replace(_PS_RE,r"\1",regex=True).str.replace(_RP_RE,r"\1",regex=True)
        return s.str.replace(_WS_RE," ",regex=True).str.strip().tolist()

# ── Dataset + bucketing ───────────────────────────────────────────────────────
class AkkadianDataset(Dataset):
    def __init__(self,df,pre,logger):
        self.ids=df["id"].tolist()
        src="transliteration" if "transliteration" in df.columns else "source"
        proc=pre.preprocess_batch(df[src].fillna("").tolist())
        self.texts=["translate Akkadian to English: "+t for t in proc]
        logger.info(f"Dataset: {len(self.ids)} samples (col='{src}')")
    def __len__(self): return len(self.ids)
    def __getitem__(self,i): return self.ids[i],self.texts[i]

class BucketBatchSampler(Sampler):
    def __init__(self,ds,bs,nb,logger):
        self.bs=bs; lengths=[len(t.split()) for _,t in ds]
        si=sorted(range(len(lengths)),key=lambda i:lengths[i]); bz=max(1,len(si)//max(1,nb))
        self.bkts=[si[i*bz:None if i==nb-1 else (i+1)*bz] for i in range(nb)]
        for i,b in enumerate(self.bkts):
            if b: logger.info(f"  Bucket {i}: {len(b)} samples, "
                              f"len[{min(lengths[x] for x in b)},{max(lengths[x] for x in b)}]")
    def __iter__(self):
        for b in self.bkts:
            for i in range(0,len(b),self.bs): yield b[i:i+self.bs]
    def __len__(self): return sum(math.ceil(len(b)/self.bs) for b in self.bkts)

# ── Model wrapper ─────────────────────────────────────────────────────────────
class ModelWrapper:
    def __init__(self,path,cfg:Config,logger,label="Model"):
        self.cfg=cfg; self.logger=logger; self.label=label
        logger.info(f"[{label}] Loading {path} …")
        self.tok=AutoTokenizer.from_pretrained(path)
        self.model=AutoModelForSeq2SeqLM.from_pretrained(path).to(DEVICE).eval()
        if DEVICE.type=="cuda":
            try: torch.set_float32_matmul_precision("high")
            except: pass
            logger.info(f"[{label}] {sum(p.numel() for p in self.model.parameters()):,} params  "
                        f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        if cfg.use_better_transformer and DEVICE.type=="cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model=BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer ✅")
            except Exception as e: logger.warning(f"[{label}] BT skipped: {e}")

    def collate(self,batch):
        ids=[b[0] for b in batch]; texts=[b[1] for b in batch]
        enc=self.tok(texts,max_length=self.cfg.max_input_length,
                     padding=True,truncation=True,return_tensors="pt")
        return ids,enc

    def _eps(self,ids,attn,eps,n):
        try:
            out=self.model.generate(input_ids=ids,attention_mask=attn,do_sample=True,
                num_beams=1,epsilon_cutoff=eps,num_return_sequences=n,
                max_new_tokens=self.cfg.max_new_tokens,
                repetition_penalty=self.cfg.repetition_penalty,use_cache=True)
            return self.tok.batch_decode(out,skip_special_tokens=True)
        except Exception as e:
            self.logger.warning(f"[{self.label}] ε={eps} failed: {e}")
            return [""]*ids.shape[0]*n

    def generate_candidates(self,ids,attn,beam_size)->List[List[str]]:
        cfg=self.cfg; B=ids.shape[0]; ctx=_bf16_ctx(DEVICE,cfg.use_bf16_amp)
        Rb=cfg.num_beam_cands; Rs=cfg.num_sample_per_temp
        with ctx:
            bout=self.model.generate(input_ids=ids,attention_mask=attn,do_sample=False,
                num_beams=max(beam_size,Rb),num_return_sequences=Rb,
                max_new_tokens=cfg.max_new_tokens,length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,use_cache=True)
            btexts=self.tok.batch_decode(bout,skip_special_tokens=True)
            ttexts=[]; nt=0
            if cfg.use_sampling:
                nt=len(cfg.sample_temperatures)
                for temp in cfg.sample_temperatures:
                    try:
                        out=self.model.generate(input_ids=ids,attention_mask=attn,
                            do_sample=True,num_beams=1,top_p=cfg.top_p,temperature=temp,
                            num_return_sequences=Rs,max_new_tokens=cfg.max_new_tokens,
                            repetition_penalty=cfg.repetition_penalty,use_cache=True)
                        ttexts.extend(self.tok.batch_decode(out,skip_special_tokens=True))
                    except Exception as e:
                        self.logger.warning(f"[{self.label}] T={temp} failed: {e}")
                        ttexts.extend([""]*B*Rs)
            Re=cfg.num_eps_cands
            etexts=self._eps(ids,attn,cfg.epsilon,Re) if Re>0 else []
        pools=[]
        for i in range(B):
            p=list(btexts[i*Rb:(i+1)*Rb])
            for t in range(nt): p.extend(ttexts[t*B*Rs+i*Rs:t*B*Rs+i*Rs+Rs])
            if etexts: p.extend(etexts[i*Re:(i+1)*Re])
            pools.append(p)
        if pools: self.logger.info(
            f"[{self.label}] hyps={len(pools[0])} (b={Rb} nuc={nt*Rs} ε={Re})")
        return pools

    def unload(self):
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model=BetterTransformer.reverse(self.model)
        except: pass
        del self.model,self.tok; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
            free=(torch.cuda.get_device_properties(0).total_memory
                  -torch.cuda.memory_allocated())/1e9
            self.logger.info(f"[{self.label}] Unloaded. GPU free: {free:.2f} GB")


# ── v20: Cross-model-aware agreement MBR ─────────────────────────────────────
class CrossModelAgreementMBR:
    """
    MBR selector that distinguishes between within-model and cross-model
    candidate agreements.

    MOTIVATION:
      When model A and model B (independently fine-tuned) both produce
      the same English string, they are effectively acting as a
      product-of-experts (PoE) distribution (Hinton 2002). The PoE
      distribution is sharper and peaks more strongly at high-quality
      translations than either individual model. This is why cross-model
      agreement is a much stronger quality signal than within-model
      repetition along adjacent beam paths.

    FORMULA:
      score(h) = count_weighted_utility(h)
               + within_model_bonus × within_extra(h)
               + cross_model_bonus  × cross_agreed(h)

      within_extra(h)  = max(0, total_count(h) - 1 - cross_agreed(h))
      cross_agreed(h)  = 1 if both model A and B produced h, else 0

      within_model_bonus = 0.015  (weak — correlated beam/sample paths)
      cross_model_bonus  = 0.12   (strong — independent model agreement)

    FALLBACK (single-model mode):
      If MODEL_B is not available, uses the competitor's uniform
      agreement_bonus = 0.035 (so the code still improves over v17).
    """

    def __init__(self, pool_cap: int, cross_bonus: float, within_bonus: float,
                 use_weighted_utility: bool = True, single_model_bonus: float = 0.035):
        self.pool_cap            = pool_cap
        self.cross_bonus         = cross_bonus
        self.within_bonus        = within_bonus
        self.use_weighted_util   = use_weighted_utility
        self.single_model_bonus  = single_model_bonus  # fallback for single-model mode
        if SACREBLEU_OK:
            self._chrf = _sacrebleu.metrics.CHRF(word_order=2)

    def _chrfpp(self, a: str, b: str) -> float:
        if not a or not b: return 0.0
        if SACREBLEU_OK:
            return float(self._chrf.sentence_score(a, [b]).score)
        return _py_chrf(a, b)

    def _sbleu(self, a: str, b: str) -> float:
        if not a or not b: return 0.0
        if SACREBLEU_OK:
            try: return float(_sacrebleu.sentence_bleu(a,[b],smooth_method='exp').score)
            except: pass
        return _py_sbleu(a, b)

    def _geomean(self, a: str, b: str) -> float:
        bl = self._sbleu(a, b); cf = self._chrfpp(a, b)
        if bl <= 0 or cf <= 0: return 0.0
        return math.sqrt(bl * cf)

    @staticmethod
    def _dedup_counts_by_model(
        cands_a: List[str], cands_b: List[str]
    ) -> Tuple[List[str], Dict[str, int], Dict[str, int]]:
        """Return (unique_ordered, count_a, count_b) tracking per-model counts."""
        count_a: Dict[str, int] = {}
        count_b: Dict[str, int] = {}
        order: List[str] = []
        for x in cands_a:
            t = str(x or "").strip()
            if not t: continue
            if t not in count_a: order.append(t); count_a[t] = 0
            count_a[t] += 1
        for x in cands_b:
            t = str(x or "").strip()
            if not t: continue
            if t not in count_b:
                if t not in count_a: order.append(t)
                count_b[t] = 0
            count_b[t] += 1
        return order, count_a, count_b

    def pick(self, cands_a: List[str], cands_b: List[str]) -> str:
        """
        Select best candidate with source-aware agreement scoring.

        cands_a: postprocessed candidates from model A
        cands_b: postprocessed candidates from model B (empty list in single-model mode)
        """
        if not cands_b:
            # Single-model fallback: use uniform agreement bonus
            return self._pick_single_model(cands_a)

        order, count_a, count_b = self._dedup_counts_by_model(cands_a, cands_b)
        if self.pool_cap: order = order[:self.pool_cap]
        n = len(order)
        if n == 0: return ""
        if n == 1: return order[0]

        # Total count per candidate (for utility weighting)
        total_counts = {c: count_a.get(c, 0) + count_b.get(c, 0) for c in order}
        total_weight = sum(total_counts[c] for c in order)

        scores = []
        for i, cand in enumerate(order):
            # Count-weighted pairwise utility (Eikema & Aziz 2022)
            if self.use_weighted_util:
                w_sum = gu_sum = 0.0
                for j, ref in enumerate(order):
                    if i == j: continue
                    w = total_counts.get(ref, 1)
                    gu_sum += self._geomean(cand, ref) * w
                    w_sum  += w
                utility = gu_sum / max(w_sum, 1e-9)
            else:
                utility = sum(
                    self._geomean(cand, order[j]) for j in range(n) if j != i
                ) / max(n-1, 1)

            # Cross-model agreement: both models produced this string
            ca = count_a.get(cand, 0); cb = count_b.get(cand, 0)
            cross_agreed = (ca > 0 and cb > 0)
            total = ca + cb

            # Decompose extras:
            #   cross-model agreement: +cross_bonus (once, regardless of how many times)
            #   remaining within-model extras: +within_bonus × count
            within_extras = max(0, total - 1 - int(cross_agreed))
            bonus = (self.cross_bonus * int(cross_agreed)
                   + self.within_bonus * within_extras)

            scores.append(utility + bonus)

        return order[int(np.argmax(scores))]

    def _pick_single_model(self, candidates: List[str]) -> str:
        """Fallback: uniform agreement bonus (competitor's approach)."""
        counts: Dict[str, int] = {}
        order: List[str] = []
        for x in candidates:
            t = str(x or "").strip()
            if not t: continue
            if t not in counts: order.append(t); counts[t] = 0
            counts[t] += 1
        if self.pool_cap: order = order[:self.pool_cap]
        n = len(order)
        if n == 0: return ""
        if n == 1: return order[0]
        scores = []
        for i, cand in enumerate(order):
            w_sum = gu_sum = 0.0
            for j, ref in enumerate(order):
                if i == j: continue
                w = counts.get(ref, 1)
                gu_sum += self._geomean(cand, ref) * w
                w_sum  += w
            utility = gu_sum / max(w_sum, 1e-9)
            bonus = self.single_model_bonus * max(0, counts.get(cand, 1) - 1)
            scores.append(utility + bonus)
        return order[int(np.argmax(scores))]


# ── Inference engine ──────────────────────────────────────────────────────────
class InferenceEngine:
    def __init__(self, cfg: Config, logger):
        self.cfg=cfg; self.logger=logger
        self.pre=OptimizedPreprocessor(); self.post=VectorizedPostprocessor()
        self.mbr=CrossModelAgreementMBR(
            pool_cap=cfg.mbr_pool_cap,
            cross_bonus=cfg.cross_model_bonus,
            within_bonus=cfg.within_model_bonus,
            use_weighted_utility=cfg.use_weighted_utility)

    def _adaptive_beams(self,attn):
        if not self.cfg.use_adaptive_beams: return self.cfg.num_beams
        med=float(attn.sum(dim=1).float().median().item())
        return max(self.cfg.num_beam_cands,self.cfg.num_beams//2) if med<100 \
               else self.cfg.num_beams

    def _dl(self,ds,wrapper):
        if self.cfg.use_bucket_batching:
            return DataLoader(ds,
                batch_sampler=BucketBatchSampler(ds,self.cfg.batch_size,
                                                 self.cfg.num_buckets,self.logger),
                num_workers=self.cfg.num_workers,collate_fn=wrapper.collate,
                pin_memory=(DEVICE.type=="cuda"))
        return DataLoader(ds,batch_size=self.cfg.batch_size,shuffle=False,
                          num_workers=self.cfg.num_workers,collate_fn=wrapper.collate,
                          pin_memory=(DEVICE.type=="cuda"))

    def _run_model(self,wrapper,ds) -> Dict[str,List[str]]:
        pools={}
        with torch.inference_mode():
            for ids,enc in tqdm(self._dl(ds,wrapper),desc=f"  {wrapper.label}"):
                idt=enc.input_ids.to(DEVICE,non_blocking=True)
                att=enc.attention_mask.to(DEVICE,non_blocking=True)
                try:
                    for sid,pool in zip(ids,wrapper.generate_candidates(
                            idt,att,self._adaptive_beams(att))):
                        pools[str(sid)]=pool
                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        self.logger.error("OOM — skipping batch")
                        torch.cuda.empty_cache()
                        for sid in ids: pools.setdefault(str(sid),[])
                    else: raise
                except Exception as e:
                    self.logger.error(f"Batch error: {e}")
                    for sid in ids: pools.setdefault(str(sid),[])
                if torch.cuda.is_available(): torch.cuda.empty_cache()
        return pools

    def run(self, test_df: pd.DataFrame) -> pd.DataFrame:
        cfg=self.cfg; log=self.logger; n_mod=2 if TWO_MODEL_MODE else 1
        log.info("="*65)
        log.info(f"v20 | {'2-Model' if TWO_MODEL_MODE else '1-Model'} | "
                 f"Cross-model-aware agreement MBR")
        log.info(f"  MODEL_A: {Path(cfg.model_a_path).relative_to(INPUT_ROOT)}")
        if TWO_MODEL_MODE:
            log.info(f"  MODEL_B: {Path(cfg.model_b_path).relative_to(INPUT_ROOT)}")
        log.info(f"  beam={cfg.num_beam_cands}×{cfg.num_beams} LP={cfg.length_penalty}")
        log.info(f"  nucleus temps={cfg.sample_temperatures} ×{cfg.num_sample_per_temp}"
                 f" = {cfg.num_sample_cands}")
        log.info(f"  ε=0.02: {cfg.num_eps_cands}  pool/model={cfg.cands_per_model}"
                 f"  total≈{cfg.cands_per_model*n_mod}")
        log.info(f"  MBR: geo-mean + cross_bonus={cfg.cross_model_bonus}"
                 f" + within_bonus={cfg.within_model_bonus}")
        log.info(f"  pool_cap={cfg.mbr_pool_cap}")
        log.info("="*65)

        ds=AkkadianDataset(test_df,self.pre,log); sids=[str(s) for s in ds.ids]

        log.info(f"Phase 1/{n_mod} — Model A")
        wa=ModelWrapper(cfg.model_a_path,cfg,log,"Model-A")
        pa=self._run_model(wa,ds); wa.unload(); del wa

        pb: Dict[str,List[str]] = {}
        if TWO_MODEL_MODE:
            log.info("Phase 2/2 — Model B")
            wb=ModelWrapper(cfg.model_b_path,cfg,log,"Model-B")
            pb=self._run_model(wb,ds); wb.unload(); del wb

        log.info("MBR: cross-model-aware agreement …")
        cross_agreements = 0
        rows=[]
        for sid in tqdm(sids,desc="  MBR"):
            raw_a = pa.get(sid, [])
            raw_b = pb.get(sid, [])
            # Postprocess each model's pool separately to preserve source identity
            pp_a = self.post.postprocess_batch(raw_a) if raw_a else []
            pp_b = self.post.postprocess_batch(raw_b) if raw_b else []

            # Log cross-model agreement rate
            set_a = set(x for x in pp_a if x.strip())
            set_b = set(x for x in pp_b if x.strip())
            n_cross = len(set_a & set_b)
            if n_cross > 0:
                cross_agreements += 1
                log.info(f"  [{sid}] {n_cross} cross-model agreement(s): "
                         f"{list(set_a & set_b)[:2]}")

            chosen = self.mbr.pick(pp_a, pp_b)
            if not chosen or not chosen.strip():
                all_pp = pp_a + pp_b
                nz = [c for c in all_pp if c.strip()]
                chosen = max(nz, key=len) if nz else "The tablet is too damaged to translate."
            rows.append((sid, chosen))

            if len(rows) % cfg.checkpoint_freq == 0:
                ckpt = Path(cfg.output_dir)/f"v20_ckpt_{len(rows)}.csv"
                pd.DataFrame(rows,columns=["id","translation"]).to_csv(ckpt,index=False)
                log.info(f"  Checkpoint {len(rows)} → {ckpt}")

        df = pd.DataFrame(rows, columns=["id","translation"])
        log.info(f"Cross-model agreements found in {cross_agreements}/{len(sids)} sentences")
        log.info(f"Done. rows={len(df)} "
                 f"empty={df['translation'].str.strip().eq('').sum()} "
                 f"avglen={df['translation'].str.len().mean():.1f}")
        for i in range(min(4,len(df))):
            log.info(f"  [{df.iloc[i]['id']}] {str(df.iloc[i]['translation'])[:90]}")
        return df


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    cfg=Config(); logger=setup_logging(cfg.output_dir,"v20")
    print(f"\nv20 — Cross-model-aware agreement MBR")
    print(f"  cross_bonus={cfg.cross_model_bonus} (both models agree)")
    print(f"  within_bonus={cfg.within_model_bonus} (same model, different path)")
    print(f"  Pool: {cfg.num_beam_cands}b + {cfg.num_sample_cands}nuc"
          f" + {cfg.num_eps_cands}ε = {cfg.cands_per_model}/model"
          f" × {1+int(TWO_MODEL_MODE)} = ~{cfg.cands_per_model*(1+int(TWO_MODEL_MODE))} total")
    print()

    test_df=pd.read_csv(cfg.test_data_path,encoding="utf-8")
    logger.info(f"Test: {len(test_df)} rows  cols={list(test_df.columns)}")
    if "id" not in test_df.columns:
        test_df=test_df.rename(columns={
            next((c for c in test_df.columns if "id" in c.lower()),test_df.columns[0]):"id"})
    if "transliteration" not in test_df.columns and "source" not in test_df.columns:
        nid=[c for c in test_df.columns if c!="id"]
        if nid: test_df=test_df.rename(columns={nid[0]:"transliteration"})

    result=InferenceEngine(cfg,logger).run(test_df)
    out=Path(cfg.output_dir)/"submission.csv"
    result.to_csv(out,index=False)
    logger.info(f"✅ {out} ({len(result)} rows)")

    with open(Path(cfg.output_dir)/"config_v20.json","w") as f:
        json.dump({"version":"v20",
            "mbr":"cross-model-aware geo-mean sqrt(sBLEU×sChrF++)",
            "cross_model_bonus":cfg.cross_model_bonus,
            "within_model_bonus":cfg.within_model_bonus,
            "use_weighted_utility":cfg.use_weighted_utility,
            "pool_cap":cfg.mbr_pool_cap,
            "models":1+int(TWO_MODEL_MODE),
            "beam":{"cands":cfg.num_beam_cands,"beams":cfg.num_beams,
                    "LP":cfg.length_penalty,"RP":cfg.repetition_penalty},
            "nucleus_temps":cfg.sample_temperatures,
            "epsilon_02":cfg.num_eps_cands,
            "pool_per_model":cfg.cands_per_model},f,indent=2)

    print(f"\n✅ submission.csv ({len(result)} rows)")
    print(result.to_string(index=False))

input/
  datasets/
    assiaben/
      final-byt5/
        byt5-akkadian-optimized-34x/
          added_tokens.json  (2 KB)
          config.json  (0 KB)
          generation_config.json  (0 KB)
          model.safetensors  (2.3 GB)
          special_tokens_map.json  (3 KB)
          tokenizer_config.json  (25 KB)
    francescocampigotto/
      offline-pkgs/
        offline_pkgs/
          peft-0.18.1-py3-none-any.whl  (543 KB)
          portalocker-3.2.0-py3-none-any.whl  (21 KB)
          sacrebleu-2.6.0-py3-none-any.whl  (98 KB)
    spencevanasperdt/
      mattiaangelibyt5-akkadian-mbrpytorchdefault6/
        added_tokens.json  (2 KB)
        config.json  (0 KB)
        generation_config.json  (0 KB)
        model.safetensors  (2.3 GB)
        special_tokens_map.json  (3 KB)
        tokenizer_config.json  (25 KB)
  competitions/
    deep-past-initiative-machine-translation/
      OA_Lexicon_eBL.csv  (3470 KB)
      Sentences_Oare_FirstWord_LinNum.csv  (1939 KB)
      bibliography.cs

2026-03-13 15:51:39,225 | INFO | Test: 4 rows  cols=['id', 'text_id', 'line_start', 'line_end', 'transliteration']
2026-03-13 15:51:39,235 | INFO | =================================================================
2026-03-13 15:51:39,236 | INFO | v20 | 2-Model | Cross-model-aware agreement MBR
2026-03-13 15:51:39,236 | INFO |   MODEL_A: datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x
2026-03-13 15:51:39,237 | INFO |   MODEL_B: datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6
2026-03-13 15:51:39,238 | INFO |   beam=4×8 LP=1.3
2026-03-13 15:51:39,239 | INFO |   nucleus temps=[0.6, 0.8, 0.75] ×2 = 6
2026-03-13 15:51:39,239 | INFO |   ε=0.02: 6  pool/model=16  total≈32
2026-03-13 15:51:39,240 | INFO |   MBR: geo-mean + cross_bonus=0.12 + within_bonus=0.015
2026-03-13 15:51:39,240 | INFO |   pool_cap=40
2026-03-13 15:51:39,241 | INFO | =================================================================
2026-03-13 15:51:39,246 | INFO | Dataset: 4 samples (col='tr

  Device: cuda / Tesla T4 / 15.6 GB

v20 — Cross-model-aware agreement MBR
  cross_bonus=0.12 (both models agree)
  within_bonus=0.015 (same model, different path)
  Pool: 4b + 6nuc + 6ε = 16/model × 2 = ~32 total



Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-13 15:52:09,045 | INFO | [Model-A] 581,653,248 params  GPU: 2.38 GB
2026-03-13 15:52:09,046 | WARNING | [Model-A] BT skipped: No module named 'optimum'
2026-03-13 15:52:09,047 | INFO |   Bucket 0: 1 samples, len[20,20]
2026-03-13 15:52:09,048 | INFO |   Bucket 1: 1 samples, len[20,20]
2026-03-13 15:52:09,049 | INFO |   Bucket 2: 1 samples, len[23,23]
2026-03-13 15:52:09,049 | INFO |   Bucket 3: 1 samples, len[38,38]


...

  Model-B:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-13 15:54:08,222 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-13 15:54:22,210 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-13 15:54:37,930 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-13 15:55:06,070 | INFO | [Model-B] hyps=16 (b=4 nuc=6 ε=6)
2026-03-13 15:55:16,610 | INFO | [Model-B] Unloaded. GPU free: 15.63 GB
2026-03-13 15:55:16,611 | INFO | MBR: cross-model-aware agreement …


  MBR:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-13 15:55:19,741 | INFO | Cross-model agreements found in 0/4 sentences
2026-03-13 15:55:19,743 | INFO | Done. rows=4 empty=0 avglen=173.8
2026-03-13 15:55:19,744 | INFO |   [0] Thus kārum Kanesh, say to the <gap> dātum, our messenger, every single colony and the trad
2026-03-13 15:55:19,745 | INFO |   [1] As for the tablet of the City, whoever buys meteoric iron, it is not indebted to the proce
2026-03-13 15:55:19,745 | INFO |   [2] As soon as you have heard our letter, whether he has sold anything there for anything, or 
2026-03-13 15:55:19,746 | INFO |   [3] I sent it to every single colony and the trading stations to every single colony and tradi
2026-03-13 15:55:19,755 | INFO | ✅ /kaggle/working/submission.csv (4 rows)



✅ submission.csv (4 rows)
id                                                                                                                                                                                                                               translation
 0                                                                                                    Thus kārum Kanesh, say to the <gap> dātum, our messenger, every single colony and the trading stations: A tablet has come to the City.
 1                                                                                   As for the tablet of the City, whoever buys meteoric iron, it is not indebted to the proceeds from the profit. The Kanesh colony will receive my tithe.
 2 As soon as you have heard our letter, whether he has sold anything there for anything, or has sold it to the palace, or has not sold it until now, write the debt of this meteoric iron in the tablet, and I shall send me our messenger.
 3                       